In [5]:
import pandas as pd

# Caminho da planilha
caminho_arquivo = r'D:\Cotic\custodiados13102025.xlsx'

# Lê a planilha
df = pd.read_excel(caminho_arquivo)

# Verifica se as colunas necessárias existem
if {'Escolaridade', 'CPF', 'Última Localização'}.issubset(df.columns):
    # Remove linhas onde 'Escolaridade' está ausente
    df_filtrado = df.dropna(subset=['Escolaridade'])

    # Lista de escolaridades a excluir
    excluir = [
        'ALFABETIZADO',
        'ENSINO FUNDAMENTAL INCOMPLETO',
        'ANALFABETO',
        'NÃO DECLARADO'
    ]
    df_filtrado = df_filtrado[~df_filtrado['Escolaridade'].str.upper().isin(excluir)]

    # Filtra linhas com CPF preenchido
    df_cpf_preenchido = df_filtrado[df_filtrado['CPF'].notna() & (df_filtrado['CPF'].astype(str).str.strip() != '')].copy()

    # Conta e exibe total de CPFs válidos
    total_cpf_preenchido = len(df_cpf_preenchido)
    print(f"Número de linhas com CPF preenchido após o filtro de escolaridade: {total_cpf_preenchido}")

    # Normaliza 'Última Localização'
    df_cpf_preenchido.loc[:, 'Última Localização'] = df_cpf_preenchido['Última Localização'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

    # Extrai "Bloco XX - Ala [Texto Possivelmente Longo com Acentos]"
    df_cpf_preenchido.loc[:, 'Bloco_Ala'] = df_cpf_preenchido['Última Localização'].str.extract(r'(Bloco \d+\s*-\s*Ala [\wÀ-ÿ\s]+)', expand=False)

    # Classifica a localização final
    def classificar_localizacao(row):
        if pd.notna(row['Bloco_Ala']):
            return row['Bloco_Ala'].strip()
        elif row['Última Localização'].lower() in ['nan', 'none', '']:
            return 'Sem Localização'
        else:
            return row['Última Localização'].strip()

    df_cpf_preenchido.loc[:, 'Localizacao_Classificada'] = df_cpf_preenchido.apply(classificar_localizacao, axis=1)

    # Conta localizações distintas
    contagem_localizacao = df_cpf_preenchido['Localizacao_Classificada'].value_counts().sort_index()

    print("\nContagem por Localização (com distinções completas):")
    print(contagem_localizacao)

else:
    print("A planilha não contém as colunas necessárias: 'Escolaridade', 'CPF' e 'Última Localização'.")


Número de linhas com CPF preenchido após o filtro de escolaridade: 657

Contagem por Localização (com distinções completas):
Localizacao_Classificada
Bloco 01 - Ala A                         21
Bloco 01 - Ala B                         41
Bloco 01 - Ala C                         41
Bloco 01 - Ala D                         50
Bloco 01 - Ala E                         59
Bloco 01 - Ala ENFERMARIA                 4
Bloco 01 - Ala F                         63
Bloco 01 - Ala G                         83
Bloco 01 - Ala H                         64
Bloco 01 - Ala I                         73
Bloco 01 - Ala K                         64
Bloco 01 - Ala SEGURANÇA A               58
Bloco 01 - Ala SEGURANÇA B               26
HOSPITAL DE SAÚDE MENTAL DE MESSEJANA     2
HOSPITAL REGIONAL NORTE                   1
Sem Localização                           7
Name: count, dtype: int64


In [6]:
import pandas as pd

# Caminho da planilha original
caminho_arquivo = r'D:\Cotic\custodiados13102025.xlsx'
# Caminho para salvar a nova planilha
caminho_saida = r'D:\Cotic\custodiados_processado.xlsx'

# Lê a planilha
df = pd.read_excel(caminho_arquivo)

# Verifica colunas necessárias
if {'Escolaridade', 'CPF', 'Última Localização'}.issubset(df.columns):
    # Filtro por escolaridade e CPF
    excluir = ['ALFABETIZADO', 'ENSINO FUNDAMENTAL INCOMPLETO', 'ANALFABETO', 'NÃO DECLARADO']
    df = df[df['Escolaridade'].notna()]
    df = df[~df['Escolaridade'].str.upper().isin(excluir)]
    df = df[df['CPF'].notna() & (df['CPF'].astype(str).str.strip() != '')].copy()

    # Lista de localizações válidas
    localizacoes_validas = [
        'Bloco 01 - Ala A',
        'Bloco 01 - Ala B',
        'Bloco 01 - Ala C',
        'Bloco 01 - Ala D',
        'Bloco 01 - Ala E',
        'Bloco 01 - Ala ENFERMARIA',
        'Bloco 01 - Ala F',
        'Bloco 01 - Ala G',
        'Bloco 01 - Ala H',
        'Bloco 01 - Ala I',
        'Bloco 01 - Ala K',
        'Bloco 01 - Ala SEGURANÇA A',
        'Bloco 01 - Ala SEGURANÇA B',
        'HOSPITAL DE SAÚDE MENTAL DE MESSEJANA',
        'HOSPITAL REGIONAL NORTE'
    ]

    # Normaliza 'Última Localização'
    df['Última Localização'] = df['Última Localização'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

    # Nova coluna: localizacaoclassificada
    def classificar_loc(ult_loc):
        for loc in localizacoes_validas:
            if ult_loc.startswith(loc):
                return loc
        return 'Sem Localização'

    df['localizacaoclassificada'] = df['Última Localização'].apply(classificar_loc)

    # Criar nova coluna endereconositeppl
    df['endereconositeppl'] = None  # Inicializa

    # Regras de divisão por ala
    regras_divisao = {
        'Bloco 01 - Ala A': [(0, None, 'Bloco 01 - Ala A - Térreo - 1')],
        'Bloco 01 - Ala C': [(0, 21, 'Bloco 01 - Ala C - Térreo - 2'),
                             (21, None, 'Bloco 01 - Ala C - Térreo - 3')],
        'Bloco 01 - Ala D': [(0, 25, 'Bloco 01 - Ala D - Térreo - 4'),
                             (25, None, 'Bloco 01 - Ala D - Térreo - 5')],
        'Bloco 01 - Ala E': [(0, 20, 'Bloco 01 - Ala E - Térreo - 6'),
                             (20, 40, 'Bloco 01 - Ala E - Térreo - 7'),
                             (40, None, 'Bloco 01 - Ala E - Térreo - 8')],
        'Bloco 01 - Ala F': [(0, 20, 'Bloco 01 - Ala F - Térreo - 9'),
                             (20, 40, 'Bloco 01 - Ala F - Térreo - 10'),
                             (40, None, 'Bloco 01 - Ala F - Térreo - 11')],
        'Bloco 01 - Ala G': [(0, 20, 'Bloco 01 - Ala G - Térreo - 12'),
                             (20, 40, 'Bloco 01 - Ala G - Térreo - 13'),
                             (40, 60, 'Bloco 01 - Ala G - Térreo - 14'),
                             (60, None, 'Bloco 01 - Ala G - Térreo - 15')],
        'Bloco 01 - Ala H': [(0, 20, 'Bloco 01 - Ala H - Térreo - 16'),
                             (20, 40, 'Bloco 01 - Ala H - Térreo - 17'),
                             (40, None, 'Bloco 01 - Ala H - Térreo - 18')],
        'Bloco 01 - Ala I': [(0, 25, 'Bloco 01 - Ala I - Térreo - 19'),
                             (25, 50, 'Bloco 01 - Ala I - Térreo - 20'),
                             (50, None, 'Bloco 01 - Ala I - Térreo - 21')],
        'Bloco 01 - Ala K': [(0, 20, 'Bloco 01 - Ala K - Térreo - 22'),
                             (20, 40, 'Bloco 01 - Ala K - Térreo - 23'),
                             (40, None, 'Bloco 01 - Ala K - Térreo - 24')],
        'Bloco 01 - Ala SEGURANÇA A': [(0, 20, 'Bloco 01 - Ala SEGURANÇA A - Térreo - 25'),
                                       (20, 40, 'Bloco 01 - Ala SEGURANÇA A - Térreo - 26'),
                                       (40, None, 'Bloco 01 - Ala SEGURANÇA A - Térreo - 27')],
    }

    # Aplica as regras de forma ordenada
    for ala, faixas in regras_divisao.items():
        df_ala = df[df['localizacaoclassificada'] == ala].copy()
        for i, (inicio, fim, endereco) in enumerate(faixas):
            if fim is None:
                mask = df_ala.iloc[inicio:].index
            else:
                mask = df_ala.iloc[inicio:fim].index
            df.loc[mask, 'endereconositeppl'] = endereco

    # Salvar no Excel
    df.to_excel(caminho_saida, index=False)
    print(f'✅ Planilha salva com sucesso em: {caminho_saida}')

else:
    print("A planilha não contém as colunas necessárias: 'Escolaridade', 'CPF' e 'Última Localização'.")


✅ Planilha salva com sucesso em: D:\Cotic\custodiados_processado.xlsx


In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time
import pandas as pd
import os

entrada_excel = r'D:\Cotic\custodiados_processado.xlsx'
df = pd.read_excel(entrada_excel)

print("Colunas encontradas na planilha:")
print(df.columns.tolist())

ja_inscritos_path = r'D:\Cotic\jaInscritos.csv'

# Lê já inscritos (se existir)
if os.path.exists(ja_inscritos_path):
    ja_inscritos = pd.read_csv(ja_inscritos_path)['Prontuário'].astype(str).tolist()
else:
    ja_inscritos = []

# Inicia o navegador
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 40)

# Acessa a página de login
driver.get("http://sistemasespeciais.inep.gov.br/unidadesprisionais/")

# Clica no botão de login externo
wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/login/div/div[3]/div/div[3]/div[1]/login-form/div/div/div[1]/a"))).click()

# Preenche o CPF e senha
wait.until(EC.presence_of_element_located((By.XPATH, "/html/body/div[2]/div[2]/div/div/div[1]/form/div[1]/input"))).send_keys("66636949368")
driver.find_element(By.XPATH, "/html/body/div[2]/div[2]/div/div/div[1]/form/div[2]/input").send_keys("Chifre2025")
driver.find_element(By.XPATH, "/html/body/div[2]/div[2]/div/div/div[1]/form/div[4]/input").click()

# Acessa a página de inscrição
wait.until(EC.url_changes("http://sistemasespeciais.inep.gov.br/unidadesprisionais/"))
driver.get("http://sistemasespeciais.inep.gov.br/unidadesprisionais/inscricao-enem")

# Aguarda carregar (demora mesmo)
time.sleep(10)

# Caminhos dos arquivos
entrada_csv = r'D:\Cotic\cpf_preenchido_ala_f.csv'
ja_inscritos_path = r'D:\Cotic\jaInscritos.csv'
erro_inscricao_path = r'D:\Cotic\erroInscricao.csv'

# Carrega os dados de entrada
df = pd.read_excel(entrada_excel)

# Carrega prontuários já inscritos
if os.path.exists(ja_inscritos_path):
    ja_inscritos = pd.read_csv(ja_inscritos_path)['Prontuário'].astype(str).tolist()
else:
    ja_inscritos = []

# Carrega prontuários que deram erro
if os.path.exists(erro_inscricao_path):
    erro_inscritos = pd.read_csv(erro_inscricao_path)['Prontuário'].astype(str).tolist()
else:
    erro_inscritos = []

total_linhas = len(df)

# Loop por todas as linhas do CSV
for index, row in df.iterrows():
    prontuario = str(row['Prontuário']).strip()
    nome = row['Nome']
    cpf = str(row['CPF']).strip()
    endereco = str(row['endereconositeppl'])
      
    def limpar_cpf(cpf_raw):
        try:
            # Converte para string
            cpf_str = str(cpf_raw).strip()

            # Remove ".0" se vier de float
            if cpf_str.endswith(".0"):
                cpf_str = cpf_str[:-2]

            # Remove qualquer caractere que não seja número
            cpf_str = ''.join(filter(str.isdigit, cpf_str))

            # Se tiver 10 dígitos, adiciona 0 à esquerda
            if len(cpf_str) == 10:
                cpf_str = '0' + cpf_str

            # Se tiver 9 dígitos, adiciona 2 0 à esquerda
            if len(cpf_str) == 9:
                cpf_str = '00' + cpf_str

            # Se tiver exatamente 11, retorna. Caso contrário, retorna None (CPF inválido)
            return cpf_str if len(cpf_str) == 11 else None

        except Exception as e:
            print(f"Erro ao limpar CPF: {cpf_raw} → {e}")
            return None

    cpf_limpo = limpar_cpf(row['CPF'])

    if not cpf_limpo:
        print(f"❌ CPF inválido: {row['CPF']} ({row['Prontuário']} - {row['Nome']})")

        # Registra no erroInscricao.csv
        with open(erro_inscricao_path, 'a', encoding='utf-8') as f:
            if os.stat(erro_inscricao_path).st_size == 0:
                f.write("Prontuário\n")
            f.write(f"{row['Prontuário']}\n")

        continue

    # Remove caracteres não numéricos (ex: pontos, traços)
    cpf = ''.join(filter(str.isdigit, cpf_limpo))

    # Corrige CPFs com menos de 11 dígitos preenchendo com zeros à esquerda
    if len(cpf) == 10:
        cpf = '0' + cpf
    elif len(cpf) < 10 or len(cpf) > 11:
        print(f"❌ CPF inválido ({cpf}) para {row['Prontuário']} - {row['Nome']}")

        # Registra erro no erroInscricao.csv
        with open(erro_inscricao_path, 'a', encoding='utf-8') as f:
            if os.stat(erro_inscricao_path).st_size == 0:
                f.write("Prontuário\n")
            f.write(f"{row['Prontuário']}\n")

        # Pula para o próximo registro
        continue


    # Pula se já inscrito ou já teve erro
    if prontuario in ja_inscritos or prontuario in erro_inscritos:
        continue

    print(f"Linha {index}/{total_linhas} - " + prontuario + " - "+ nome + " - " +cpf)
    print(endereco)

    try:
        # Clica no botão "Nova inscrição"
        wait.until(EC.element_to_be_clickable((By.XPATH,
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-listar/div/div[2]/div/mat-expansion-panel/div/div/div[3]/button"
        ))).click()

        # Preenche o CPF
        wait.until(EC.presence_of_element_located((By.XPATH,
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[1]/app-inscricao-dados-participante/div/form/mat-card[1]/mat-card-content/cpf/mat-form-field/div/div[1]/div/input"
        ))).send_keys(cpf)

        def selecionar_opcao_por_texto_parcial(parte_texto):
            xpath_opcao = f"//mat-option/span[contains(normalize-space(), '{parte_texto}')]"
            wait.until(EC.element_to_be_clickable((By.XPATH, xpath_opcao))).click()


        # Realiza seleção nos campos do formulário
        def selecionar(xpath_menu, xpath_opcao):
            wait.until(EC.element_to_be_clickable((By.XPATH, xpath_menu))).click()
            wait.until(EC.element_to_be_clickable((By.XPATH, xpath_opcao))).click()
            time.sleep(0.5)

        def textoOpcao(xpath_menu, texto_opcao):
            esperar_overlay_sumir()
            # Abre o menu dropdown
            click_com_retry(By.XPATH, xpath_menu)
            esperar_overlay_sumir()
            # Monta o XPath baseado no texto visível da opção
            xpath_opcao = f"//mat-option/span[contains(., '{texto_opcao}')]"

            # Aguarda e clica na opção desejada
            click_com_retry(By.XPATH, xpath_opcao)
            time.sleep(0.5)

        def esperar_overlay_sumir(timeout=15):
           try:
               WebDriverWait(driver, timeout).until_not(
                   EC.presence_of_element_located((By.CSS_SELECTOR, ".ngx-overlay"))
               )
               WebDriverWait(driver, timeout).until_not(
                   EC.presence_of_element_located((By.CSS_SELECTOR, ".cdk-overlay-backdrop"))
               )
           except TimeoutException:
               print("⚠️ Overlay (ngx ou cdk) não desapareceu dentro do tempo esperado.")

        def click_com_retry(by, value, tentativas=5):
            for i in range(tentativas):
                try:
                    esperar_overlay_sumir()
                    el = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((by, value)))
                    driver.execute_script("arguments[0].scrollIntoView(true);", el)
                    el.click()
                    return True
                except Exception as e:
                    print(f"⚠️ Tentativa {i+1} falhou: {str(e)}")
                    time.sleep(1)
            return False


        selecionar(
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[1]/app-inscricao-dados-participante/div/form/mat-card[2]/mat-card-content/mat-form-field[2]/div/div[1]/div/mat-select/div/div[1]",
            "/html/body/div[4]/div[2]/div/div/div/mat-option[3]/span"
        )

        textoOpcao(
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[1]/app-inscricao-dados-participante/div/form/mat-card[2]/mat-card-content/div/mat-form-field/div/div[1]/div/mat-select/div/div[1]",
            endereco
        )

        selecionar(
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[1]/app-inscricao-dados-participante/div/form/mat-card[3]/mat-card-content/mat-form-field/div/div[1]/div/mat-select/div/div[1]",
            "/html/body/div[4]/div[2]/div/div/div/mat-option[3]/span"
        )

        selecionar(
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[1]/app-inscricao-dados-participante/div/form/mat-card[4]/mat-card-content/mat-form-field/div/div[1]/div/mat-select/div/div[1]",
            "/html/body/div[4]/div[2]/div/div/div/mat-option[3]/span"
        )

        

        # Avançar etapas
        botoes = [
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[1]/div/button[2]/span",
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[2]/div/button[2]/span",
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[3]/div/button[2]/span",
            "/html/body/app/vertical-layout-1/div/div[2]/div/div/content/app-inscricao-form/div/div[2]/div/mat-horizontal-stepper/div[2]/div[4]/div/button[2]/span"
        ]

        for botao in botoes:
            wait.until(EC.element_to_be_clickable((By.XPATH, botao))).click()
            time.sleep(1)

        # Salva prontuário como concluído
        with open(ja_inscritos_path, 'a', encoding='utf-8') as f:
            if os.stat(ja_inscritos_path).st_size == 0:
                f.write("Prontuário\n")
            f.write(f"{prontuario}\n")

        print(f"Inscrição de {prontuario} - {nome} concluída.")
        click_com_retry(By.XPATH, "/html/body/div[4]/div[2]/div/mat-dialog-container/app-dialog-alert/div[3]/div/button/span")
    
    except Exception as e:
        print(f"❌ Erro com {prontuario} - {nome}: {e}")

        # Salva prontuário com erro
        with open(erro_inscricao_path, 'a', encoding='utf-8') as f:
            if os.stat(erro_inscricao_path).st_size == 0:
                f.write("Prontuário\n")
            f.write(f"{prontuario}\n")

        # Volta para a tela de início de inscrição
        driver.get("http://sistemasespeciais.inep.gov.br/unidadesprisionais/inscricao-enem")
        time.sleep(10)
        esperar_overlay_sumir()

        continue

# Finaliza
driver.quit()


Colunas encontradas na planilha:
['Prontuário', 'Nome', 'CPF', 'Mãe', 'Escolaridade', 'Unidade', 'Última Localização', 'Tipo de Regime', 'localizacaoclassificada', 'endereconositeppl']
Linha 110/650 - 457113 - FRANCISCO JONATAS CARNEIRO - 62330422326
Bloco 01 - Ala E - Térreo – 6
⚠️ Overlay (ngx ou cdk) não desapareceu dentro do tempo esperado.
⚠️ Overlay (ngx ou cdk) não desapareceu dentro do tempo esperado.
⚠️ Overlay (ngx ou cdk) não desapareceu dentro do tempo esperado.
⚠️ Overlay (ngx ou cdk) não desapareceu dentro do tempo esperado.
⚠️ Tentativa 1 falhou: Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff7df22e9e5+80021]
	GetHandleVerifier [0x0x7ff7df22ea40+80112]
	(No symbol) [0x0x7ff7defb060f]
	(No symbol) [0x0x7ff7df008854]
	(No symbol) [0x0x7ff7df008b1c]
	(No symbol) [0x0x7ff7df05c927]
	(No symbol) [0x0x7ff7df03126f]
	(No symbol) [0x0x7ff7df05968a]
	(No symbol) [0x0x7ff7df031003]
	(No symbol) [0x0x7ff7deff95d1]
	(No symbol) [0x0x7ff7deffa3f3]
	GetHandleVerifier [0x0x7ff7df4edd

In [11]:
import pandas as pd
import re

# Carregar o arquivo Excel
caminho_saida = r'D:\Cotic\custodiados_processado.xlsx'
df = pd.read_excel(caminho_saida)

# Função para extrair o número da sala do final da string
def extrair_sala(endereco):
    # Usa regex para pegar o número no final da string
    match = re.search(r'(\d+)$', endereco)
    return int(match.group(1)) if match else None

# Aplicar a função para extrair o número da sala
df['sala'] = df['endereconositeppl'].apply(extrair_sala)

# Contar a quantidade de registros por sala
quantidade_salas = df['sala'].value_counts().sort_index()

# Exibir o resultado
for sala, quantidade in quantidade_salas.items():
    print(f"Sala {sala}: {quantidade}")

# Exibir total de inscritos
total_inscritos = df.shape[0]
print(f"\nTotal de inscritos no Enem: {total_inscritos}")


Sala 1: 23
Sala 2: 21
Sala 3: 20
Sala 4: 25
Sala 5: 25
Sala 6: 26
Sala 7: 26
Sala 8: 23
Sala 9: 20
Sala 10: 20
Sala 11: 23
Sala 12: 23
Sala 13: 22
Sala 14: 22
Sala 15: 23
Sala 16: 20
Sala 17: 20
Sala 18: 24
Sala 19: 25
Sala 20: 25
Sala 21: 23
Sala 22: 20
Sala 23: 20
Sala 24: 24
Sala 25: 25
Sala 26: 22
Sala 27: 20
Sala 28: 22
Sala 29: 24

Total de inscritos no Enem: 656
